# Model L (Nepali) — Kaggle Pretraining Notebook

In [1]:
import os
from pathlib import Path

def find_bundle_root(base="/kaggle/input", need=("NEPALI", "src")):
    for root, dirs, files in os.walk(base):
        if all(n in dirs for n in need):
            return Path(root)
    raise FileNotFoundError(
        f"Couldn't find a folder containing {need} under {base}. "
        "Run `!find /kaggle/input -maxdepth 3` to inspect what got mounted, "
        "and confirm the dataset is attached."
    )

INPUT_ROOT = find_bundle_root()
print("Bundle found at:", INPUT_ROOT)

Bundle found at: /kaggle/input/datasets/ervarishitha11/phase2-nepali


## Build a writable working copy
Symlinks the read-only tokenizer/data (and `src/`, `tests/`) straight out of the mounted dataset instead of copying several GB, and creates a real writable `outputs/` dir for checkpoints/cache/report.

In [2]:
import shutil

WORK = Path("/kaggle/working/phase2")
if WORK.exists():
    shutil.rmtree(WORK)
WORK.mkdir(parents=True)

shutil.copytree(INPUT_ROOT / "src", WORK / "src")  # copied, not symlinked -- src/ is tiny and this way it can be overwritten in-session if a fix lands after the dataset was uploaded
os.symlink(INPUT_ROOT / "tests", WORK / "tests")

(WORK / "NEPALI").mkdir()
os.symlink(INPUT_ROOT / "NEPALI" / "PHASE1", WORK / "NEPALI" / "PHASE1")  # tokenizer + data splits, read-only, fine
(WORK / "NEPALI" / "PHASE2").mkdir()
os.symlink(INPUT_ROOT / "NEPALI" / "PHASE2" / "configs", WORK / "NEPALI" / "PHASE2" / "configs")
for s in ["train_nepali.py", "eval_nepali.py"]:
    shutil.copy(INPUT_ROOT / "NEPALI" / "PHASE2" / s, WORK / "NEPALI" / "PHASE2" / s)
(WORK / "NEPALI" / "PHASE2" / "outputs").mkdir(parents=True)  # writable: checkpoints/cache/report

print("Ready:")
os.system(f"find {WORK} -maxdepth 3 | sort")

Ready:
/kaggle/working/phase2
/kaggle/working/phase2/NEPALI
/kaggle/working/phase2/NEPALI/PHASE1
/kaggle/working/phase2/NEPALI/PHASE2
/kaggle/working/phase2/NEPALI/PHASE2/configs
/kaggle/working/phase2/NEPALI/PHASE2/eval_nepali.py
/kaggle/working/phase2/NEPALI/PHASE2/outputs
/kaggle/working/phase2/NEPALI/PHASE2/train_nepali.py
/kaggle/working/phase2/src
/kaggle/working/phase2/src/attention_analyzer.py
/kaggle/working/phase2/src/dataset.py
/kaggle/working/phase2/src/generate_report_visuals.py
/kaggle/working/phase2/src/__init__.py
/kaggle/working/phase2/src/metrics.py
/kaggle/working/phase2/src/model.py
/kaggle/working/phase2/src/trainer.py
/kaggle/working/phase2/tests


0

## Install dependencies + connect Google Drive (rclone)

In [3]:
os.system("pip install -q sentencepiece sacrebleu rouge-score")
os.system("curl https://rclone.org/install.sh | sudo bash")

from kaggle_secrets import UserSecretsClient
conf_dir = os.path.expanduser("~/.config/rclone")
os.makedirs(conf_dir, exist_ok=True)

raw_secret = UserSecretsClient().get_secret("RCLONE_CONF")
# Kaggle's Secret value field flattens real newlines into spaces, so the config must be
# stored with newlines encoded as literal \n -- decode that back into real line breaks
# before writing, or rclone can't find the [gdrive] section.
conf_content = raw_secret.replace("\\n", "\n")
with open(f"{conf_dir}/rclone.conf", "w") as f:
    f.write(conf_content)

os.system("rclone lsd gdrive:")  # sanity check -- should list your Drive's top-level folders

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 6.3 MB/s eta 0:00:00


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  4734  100  4734    0     0   9543      0 --:--:-- --:--:-- --:--:--  9544


Archive:  rclone-current-linux-amd64.zip
   creating: tmp_unzip_dir_for_rclone/rclone-v1.75.0-linux-amd64/
  inflating: tmp_unzip_dir_for_rclone/rclone-v1.75.0-linux-amd64/README.html  [text]  
  inflating: tmp_unzip_dir_for_rclone/rclone-v1.75.0-linux-amd64/README.txt  [text]  
  inflating: tmp_unzip_dir_for_rclone/rclone-v1.75.0-linux-amd64/rclone  [binary]
  inflating: tmp_unzip_dir_for_rclone/rclone-v1.75.0-linux-amd64/rclone.1  [text]  
  inflating: tmp_unzip_dir_for_rclone/rclone-v1.75.0-linux-amd64/git-log.txt  [text]  


Updating index cache for path `/usr/local/man/man1'. Wait...
done.


Purging old database entries in /usr/share/man...
Processing manual pages under /usr/share/man...
Purging old database entries in /usr/share/man/pl...
Processing manual pages under /usr/share/man/pl...
Purging old database entries in /usr/share/man/pt_BR...
Processing manual pages under /usr/share/man/pt_BR...
Purging old database entries in /usr/share/man/tr...
Processing manual pages under /usr/share/man/tr...
Purging old database entries in /usr/share/man/ko...
Processing manual pages under /usr/share/man/ko...
Purging old database entries in /usr/share/man/fr...
Processing manual pages under /usr/share/man/fr...
Purging old database entries in /usr/share/man/da...
Processing manual pages under /usr/share/man/da...
Purging old database entries in /usr/share/man/ja...
Processing manual pages under /usr/share/man/ja...
Purging old database entries in /usr/share/man/es...
Processing manual pages under /usr/share/man/es...
Purging old database entries in /usr/share/man/de...
Processing 

2026/09/03 15:28:02 NOTICE: gdrive: This remote uses rclone's shared Google Drive client_id, which is being retired and will stop working during 2026. Create your own client_id to avoid interruption: https://rclone.org/drive/#making-your-own-client-id


           0 2026-08-21 05:27:42        -1 2025201027_LMA_Phase1
           0 2026-08-16 08:50:01        -1 Colab Notebooks
           0 2026-09-02 14:24:00        -1 LMA_Project1


0

## Resume safety: pull down any checkpoint from a previous session
(No-op the very first time.)

In [4]:
os.system(f"rclone copy gdrive:LMA_Project1/NEPALI_checkpoints {WORK}/NEPALI/PHASE2/outputs/checkpoints -P")

2026/09/03 15:28:36 NOTICE: gdrive: This remote uses rclone's shared Google Drive client_id, which is being retired and will stop working during 2026. Create your own client_id to avoid interruption: https://rclone.org/drive/#making-your-own-client-id


Transferred:   	          0 B / 0 B, -, 0 B/s, ETA -
Elapsed time:         0.1s


0

## Start the background Drive-sync loop
Syncs every 5 minutes for the life of the session, so at most ~5 minutes of work is ever at risk if the session dies without warning.

In [5]:
sync_sh = f"""
while true; do
  rclone sync {WORK}/NEPALI/PHASE2/outputs/checkpoints gdrive:LMA_Project1/NEPALI_checkpoints --include \"checkpoint_latest.pt\" --include \"checkpoint_final.pt\" --include \"checkpoint_step_*.pt\" --include \"training_history.json\"
  sleep 300
done
"""
open("/kaggle/working/sync_nepali.sh", "w").write(sync_sh)
os.system("nohup bash /kaggle/working/sync_nepali.sh > /kaggle/working/sync_nepali.log 2>&1 &")
print("Background Drive sync started.")

Background Drive sync started.


## Sanity check before spending GPU hours
Confirms the GPU is visible and the model/tokenizer/checkpoint pipeline loads correctly against the working bundle.

In [6]:
import torch
print("CUDA available:", torch.cuda.is_available(), "-",
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")

os.chdir(str(WORK))

import sys
sys.path.insert(0, str(WORK))
from src.model import GPTLanguageModel
from src.dataset import SentencePieceTokenizer

model = GPTLanguageModel(vocab_size=12000, d_model=512, n_layer=6, n_head=8, d_ff=2048,
                          max_seq_len=512, dropout=0.1, tie_weights=True)
params = model.count_parameters()
print("Total trainable parameters:", f"{params['total_trainable']:,}")
assert 24_000_000 < params['total_trainable'] < 26_000_000
assert model.verify_causal_masking(), "causal mask check failed"
print("Causal masking check: PASSED")

tok = SentencePieceTokenizer(str(WORK / "NEPALI/PHASE1/outputs/tokenizer/nepali_bpe.model"))
print("Tokenizer vocab size:", tok.vocab_size)
print("Sample roundtrip:", tok.decode(tok.encode("नेपाल एक देश हो")))
print("\nALL SANITY CHECKS PASSED")

CUDA available: True - Tesla T4
Total trainable parameters: 25,321,472
Causal masking check: PASSED
Tokenizer vocab size: 12000
Sample roundtrip: नेपाल एक देश हो

ALL SANITY CHECKS PASSED


## Pretrain Model L (Nepali)
Auto-resumes from `checkpoint_latest.pt` if one was pulled down in the resume-safety cell above.

In [7]:
%cd {WORK}
!python NEPALI/PHASE2/train_nepali.py

/kaggle/working
=== Starting Model L (Nepali) Pretraining ===
Config: {'language': 'nepali', 'model_tier': 'Model L', 'vocab_size': 12000, 'd_model': 512, 'n_layer': 6, 'n_head': 8, 'd_ff': 2048, 'max_seq_len': 512, 'dropout': 0.1, 'tie_weights': True, 'learning_rate': 0.0006, 'weight_decay': 0.1, 'betas': [0.9, 0.95], 'batch_size': 32, 'max_steps': 30500, 'warmup_steps': 1000, 'grad_clip': 1.0, 'eval_interval': 500, 'save_interval': 1000, 'milestone_interval': 5000, 'keep_last_n_checkpoints': 2}
Loaded Nepali tokenizer. Vocab size: 12,000
[TextDataset] Cached 1,174,677 docs / 523,698,453 tokens -> train.v12000.dall.ids.bin (2094.8 MB)
[TextDataset] Cached 5,932 docs / 2,688,965 tokens -> val.v12000.dall.ids.bin (10.8 MB)
Model Parameters: Total = 25,321,472 | Embedding = 6,406,144 | Blocks = 18,914,304
Causal mask verification test PASSED!
[Trainer] Starting training on cuda. Target steps: 30500
Step      1/30500 | Train Loss: 9.4819 | Val Loss: 9.4888 | PPL: 13210.39 | LR: 0.000001
S

## Evaluate the trained checkpoint, then push everything to Drive
Run once `checkpoint_final.pt` exists (training reached `max_steps`), or any time for an interim eval against `checkpoint_latest.pt`.

In [8]:
!python NEPALI/PHASE2/eval_nepali.py

os.system(f"rclone sync {WORK}/NEPALI/PHASE2/outputs/checkpoints gdrive:LMA_Project1/NEPALI_checkpoints -P")
os.system(f"rclone copy {WORK}/NEPALI/PHASE2/outputs/report gdrive:LMA_Project1/NEPALI_checkpoints/report -P")
print("Done -- Nepali checkpoints, logs, and eval report are all on Drive.")

=== Evaluating Model L (Nepali) on cuda ===
Loading checkpoint from /kaggle/working/phase2/NEPALI/PHASE2/outputs/checkpoints/checkpoint_final.pt

--- Intrinsic LM Metrics ---
Val Loss:   3.5140
Perplexity: 33.5828
BPB:        0.4284

--- Extracted 10 Prefix/Reference Pairs from test.jsonl ---

Prompt: प्राईड नेपालको आयोजनामा प्रस्तुत “नव वर्ष 2073 साँस्कृतिक साँझ”
  [Temp 0.5]: प्राईड नेपालको आयोजनामा प्रस्तुत “नव वर्ष 2073 साँस्कृतिक साँझ” को अन्तिम दिन “नव वर्ष 2075” को अन्तिम दिन “नव
  [Temp 1.0]: प्राईड नेपालको आयोजनामा प्रस्तुत “नव वर्ष 2073 साँस्कृतिक साँझ”को शुक्रबार भुक्तानी आयोजकले गर्ने भएको छ । गत वैशाख 26 गतेदेखी
  [Temp 1.5]: प्राईड नेपालको आयोजनामा प्रस्तुत “नव वर्ष 2073 साँस्कृतिक साँझ” युरोप संकलन संकल /तोईस जु Fmp पवित्र हुनेछु

Prompt: काठमाडौंबाट मुम्बई उड्न लागेको विमानबाट यात्रु ओरालिएको छ । वि
  [Temp 0.5]: काठमाडौंबाट मुम्बई उड्न लागेको विमानबाट यात्रु ओरालिएको छ । विमानमा चढ्ने यात्रुको संख्या करिब 50 पुगेको छ । विमानले विमानमा
  [Temp 1.0]: काठमाडौंबाट मुम्बई उ

2026/09/03 20:05:03 NOTICE: gdrive: This remote uses rclone's shared Google Drive client_id, which is being retired and will stop working during 2026. Create your own client_id to avoid interruption: https://rclone.org/drive/#making-your-own-client-id


Transferred:   	          0 B / 0 B, -, 0 B/s, ETA -
Checks:                 5 / 5, 100%, Listed 10
Elapsed time:         0.2s


2026/09/03 20:05:38 NOTICE: gdrive: This remote uses rclone's shared Google Drive client_id, which is being retired and will stop working during 2026. Create your own client_id to avoid interruption: https://rclone.org/drive/#making-your-own-client-id


Transferred:   	          0 B / 438.817 KiB, 0%, 0 B/s, ETA -
Checks:                 0 / 0, -, Listed 5
Transferred:            0 / 5, 0%
Elapsed time:         0.5s
Transferring:
 *                         attn_layer1_head1.png:  0% / 67.259 KiB, 0 B/s, -
 *                attn_layer1_multihead_grid.png:  0% / 141.039 KiB, 0 B/s, -
 *                         attn_layer6_head1.png:  0% / 68.456 KiB, 0 B/s, -
 *                attn_layer6_multihead_grid.png:  0% / 141.014 KiB, 0 B/s, -Transferred:   	          0 B / 438.817 KiB, 0%, 0 B/s, ETA -
Checks:                 0 / 0, -, Listed 5
Transferred:            0 / 5, 0%
Elapsed time:         1.0s
Transferring:
 *                         attn_layer1_head1.png:  0% / 67.259 KiB, 0 B/s, -
 *                attn_layer1_multihead_grid.png:  0% / 141.039 KiB, 0 B/s, -
 *                         attn_layer6_head1.png:  0% / 68.456 KiB, 0 B/s, -
 *                attn_layer6_multihead_grid.png:  0% / 141.014 KiB, 0 B/s, -Transferred:   	   67.

## Generate report figures and push to Drive

In [9]:
!python src/generate_report_visuals.py

[Visuals] Architecture diagram saved -> /kaggle/working/phase2/report/figures/architecture_diagram.png
[Visuals] Corpus comparison chart saved -> /kaggle/working/phase2/report/figures/corpus_comparison.png
[Visuals] Convergence curve saved -> /kaggle/working/phase2/report/figures/pretrain_loss_curve.png
[Visuals] PPL curve (log scale) saved -> /kaggle/working/phase2/report/figures/pretrain_ppl_curve_logscale.png
[Visuals] LR schedule saved -> /kaggle/working/phase2/report/figures/lr_schedule.png
[Visuals] Throughput chart saved -> /kaggle/working/phase2/report/figures/training_throughput.png
[Visuals] Attention Entropy by Layer and Head saved -> /kaggle/working/phase2/report/figures/attention_entropy_heatmap.png
[Visuals] Mean Attention Distance by Layer and Head saved -> /kaggle/working/phase2/report/figures/attention_distance_heatmap.png
[Visuals] Generation quality bar chart saved -> /kaggle/working/phase2/report/figures/generation_quality_comparison.png
[Visuals] Diversity bar char